# SMTC Accounts — EDA v2 (full dataset)

**Source:** `accounts.csv` — Salesforce **Account** export (~157K rows × 443 columns).
**Purpose:** decide (1) which columns become **Critical Data Elements**, (2) which **rows are in scope**,
(3) what **rules** to write, and (4) how to make this run at **5–10M rows**.

**What's new vs v1**
1. **Column-pruning memory benchmark** — the scaling answer
2. **Junk-column detector** (HTML/UI fields masquerading as data)
3. **Scope analysis** — the "Temporary" record problem
4. **Near-duplicate category detector** (`Middle East` vs `MiddleEast`)
5. **CDE candidate ranking** — a shortlist to take to governance
6. **Violation-volume estimate** — sanity-check the 30–40% assumption
7. **Fixed** date detection (v1 matched `CreatedById` / `SendMailTeamUpd**ate**__c`)

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110; plt.rcParams["axes.titleweight"] = "bold"
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. Load

In [ ]:
CANDIDATES = ["accounts.csv", "temp.csv",
              os.path.expanduser("~/Desktop/accounts.csv"),
              os.path.expanduser("~/Desktop/temp.csv")]
CSV_PATH = next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
print("Reading:", CSV_PATH)

df = pd.read_csv(CSV_PATH, low_memory=False)
n_rows, n_cols = df.shape
print(f"Shape: {n_rows:,} rows x {n_cols:,} columns")
df.head(3)

In [ ]:
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory (all columns) : {mem_mb:,.1f} MB")
print(f"Memory per row       : {mem_mb*1024/n_rows:,.2f} KB")
print(f"Fully duplicated rows: {df.duplicated().sum():,}")
print("\nDtypes:"); print(df.dtypes.value_counts())

### 1a. Scaling projection — why this is the #1 architecture finding

In [ ]:
per_row_kb = mem_mb*1024/n_rows
proj = pd.DataFrame({"rows": [n_rows, 1_000_000, 5_000_000, 10_000_000]})
proj["est_memory_GB_all_columns"] = (proj["rows"] * per_row_kb / 1024**2).round(1)
proj["verdict"] = np.where(proj["est_memory_GB_all_columns"] > 16, "won't fit in RAM", "ok")
print(f"Measured: {per_row_kb:.2f} KB per row across all {n_cols} columns\n")
proj

## 2. Missing-value analysis

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
fill = 1 - miss
fully_empty, fully_full = miss[miss == 1.0], miss[miss == 0.0]
print(f"100% EMPTY columns : {len(fully_empty):>3} ({len(fully_empty)/n_cols:.0%})")
print(f"100% FULL columns  : {len(fully_full):>3} ({len(fully_full)/n_cols:.0%})")
print(f"Partially filled   : {n_cols-len(fully_empty)-len(fully_full):>3}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
sns.histplot(fill.values, bins=25, ax=ax[0], color="#2f5fd0")
ax[0].set_title(f"Fill-rate distribution ({n_cols} columns)")
ax[0].set_xlabel("fraction of rows populated"); ax[0].set_ylabel("columns")
bc = pd.cut(fill, [-.01, 0, .999, 1.0],
            labels=["100% empty", "partial", "100% full"]).value_counts()
bc = bc.reindex(["100% empty", "partial", "100% full"])
sns.barplot(x=bc.index, y=bc.values, ax=ax[1], palette=["#cf3b3b", "#c98510", "#1f9d57"])
ax[1].set_title("Column completeness buckets"); ax[1].set_ylabel("columns")
for i, v in enumerate(bc.values): ax[1].text(i, v+2, str(v), ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Junk-column detector (Salesforce UI fields, not real data)

In [ ]:
def junk_kind(s):
    sample = s.dropna().astype(str).head(200)
    if sample.empty: return None
    if sample.str.contains(r"<a href|<img |</a>", regex=True).mean() > .5: return "HTML markup"
    if sample.str.fullmatch(r"[A-Za-z0-9+/]{16,}={0,2}").mean() > .5:      return "base64 blob"
    if sample.str.startswith("/services/").mean() > .5:                    return "internal URL"
    if sample.str.startswith("http").mean() > .9 and s.nunique() > len(s)*.9: return "per-row URL"
    return None

junk = {c: k for c in df.columns if (k := junk_kind(df[c]))}
junk_s = pd.Series(junk, name="kind")
print(f"Junk/UI columns detected: {len(junk_s)}")
if len(junk_s):
    junk_mem = df[list(junk_s.index)].memory_usage(deep=True).sum()/1024**2
    print(f"Memory consumed by junk columns: {junk_mem:,.1f} MB "
          f"({junk_mem/mem_mb:.0%} of total)\n")
    display(junk_s.to_frame())

## 4. Column classification (empty / constant / key / categorical / numeric)

In [ ]:
df_pop = df.dropna(axis=1, how="all")
card = df_pop.nunique(dropna=True)
klass = {}
for c in df.columns:
    if c in fully_empty.index:        klass[c] = "empty"
    elif c in junk:                   klass[c] = "junk/UI"
    elif card.get(c, 0) <= 1:         klass[c] = "constant"
    elif card.get(c, 0) == len(df):   klass[c] = "unique key"
    elif card.get(c, 0) <= 25:        klass[c] = "categorical"
    elif pd.api.types.is_numeric_dtype(df[c]): klass[c] = "numeric"
    else:                             klass[c] = "high-cardinality text"
kl = pd.Series(klass).value_counts()

plt.figure(figsize=(8, 4))
sns.barplot(x=kl.values, y=kl.index, palette="crest")
plt.title("Column classification"); plt.xlabel("number of columns")
for i, v in enumerate(kl.values): plt.text(v+1, i, str(v), va="center", fontweight="bold")
plt.tight_layout(); plt.show()

print("Real unique-key candidates (excluding junk):")
print([c for c in df.columns if klass[c] == "unique key"])

## 5. Scope analysis — the 'Temporary' record problem

In [ ]:
SCOPE_COL = "Customer_Status__c"
if SCOPE_COL in df.columns:
    vc = df[SCOPE_COL].value_counts(dropna=False)
    print(vc.to_string()); print()
    temp_mask = df[SCOPE_COL].astype(str).str.strip().str.lower().eq("temporary")
    print(f"Temporary rows : {temp_mask.sum():,} ({temp_mask.mean():.1%})")
    print(f"In-scope rows  : {(~temp_mask).sum():,} ({(~temp_mask).mean():.1%})")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax[0], palette="flare")
    ax[0].set_title("Customer_Status__c"); ax[0].set_xlabel("rows")
    ax[1].pie([temp_mask.sum(), (~temp_mask).sum()],
              labels=["Temporary\n(exclude?)", "Real accounts\n(in scope)"],
              autopct="%1.1f%%", colors=["#cf3b3b", "#1f9d57"], startangle=90)
    ax[1].set_title("Validation scope decision")
    plt.tight_layout(); plt.show()

    df_scope = df.loc[~temp_mask].copy()
    print(f"\n-> df_scope holds {len(df_scope):,} rows for scoped checks below.")
else:
    df_scope = df.copy(); temp_mask = pd.Series(False, index=df.index)

**Why this matters:** validating the Temporary rows drags every dashboard KPI toward a number
nobody can act on. Decision needed: exclude them, or expose status as a dashboard filter.

## 6. Categorical deep-dive + inconsistent-value detector

In [ ]:
cat_focus = [c for c in ["Type","Customer_Status__c","CurrencyIsoCode","Country_Region__c",
                         "Country_Sub_Region__c","Region_Name__c","Sales_Channel__c",
                         "Customer_Segmentation__c","Corporate_Account_Type__c",
                         "Service_Level__c","US_Govt_Sanctioned__c"] if c in df.columns]
for c in cat_focus:
    print(f"\n=== {c} ({df[c].nunique()} distinct, {df[c].isna().mean():.1%} null) ===")
    print(df[c].value_counts(dropna=False).head(12).to_string())

In [ ]:
# Near-duplicate category values: same string once punctuation/case/space removed
def norm(v): return re.sub(r"[^a-z0-9]", "", str(v).lower())

problems = []
low_card = [c for c in df.columns if 1 < df[c].nunique(dropna=True) <= 60]
for c in low_card:
    grp = {}
    for v in df[c].dropna().unique():
        grp.setdefault(norm(v), []).append(v)
    for k, variants in grp.items():
        if len(variants) > 1:
            counts = {v: int((df[c] == v).sum()) for v in variants}
            problems.append((c, " | ".join(f"'{v}'x{n:,}" for v, n in counts.items())))

print(f"Columns with near-duplicate category values: {len(problems)}")
pd.DataFrame(problems, columns=["column", "variants (same value, different spelling)"]).head(20)

## 7. Data-quality checks

In [ ]:
checks = []
def add(name, bad, total, note=""):
    checks.append({"check": name, "violations": bad, "population": total,
                   "rate": f"{(bad/total*100):.1f}%" if total else "-", "note": note})

if "Id" in df: add("Duplicate primary key (Id)", int(df["Id"].duplicated().sum()), n_rows, "must be 0")
if "Name" in df:
    add("Rows sharing a Name", int(df["Name"].duplicated(keep=False).sum()), n_rows, "possible dup accounts")
    add("Name is blank", int(df["Name"].isna().sum()), n_rows, "completeness")

nv = [c for c in ["Name","SAP_Account_Name__c","REPORT_ACCOUNT_NAME__c"] if c in df.columns]
if len(nv) >= 2:
    both = df[nv].dropna(how="any")
    add(f"Name mismatch across {len(nv)} name fields",
        int((both.nunique(axis=1) > 1).sum()), len(both), "cross-field consistency")

if "Duns_Number__c" in df:
    d = df["Duns_Number__c"].dropna().astype(str).str.replace(r"\.0$", "", regex=True)
    add("DUNS not 9 digits", int((~d.str.fullmatch(r"\d{9}")).sum()), len(d), "format")
if "Website" in df:
    w = df["Website"].dropna().astype(str)
    add("Website malformed", int((~w.str.match(r"^(https?://|www\.)", case=False)).sum()), len(w), "format")
for col in ["Pipeline_Amount_Total__c", "Total_Closed_Amount__c"]:
    if col in df:
        s = pd.to_numeric(df[col], errors="coerce")
        add(f"{col} negative", int((s < 0).sum()), int(s.notna().sum()), "range")

res = pd.DataFrame(checks)
display(res)

plot = res[res.violations > 0].sort_values("violations")
plt.figure(figsize=(9, max(3, .5*len(plot))))
sns.barplot(x="violations", y="check", data=plot, palette="rocket")
plt.title("Violations found by check"); plt.xlabel("records"); plt.ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
# Concrete examples of the name-mismatch problem
if len(nv) >= 3:
    both = df[nv].dropna(how="any")
    bad = both[both.nunique(axis=1) > 1]
    print(f"{len(bad):,} records where the three name fields disagree. Examples:")
    display(bad.head(10))

### 7a. Date columns — detected by VALUE, not by column name (v1 bug fixed)

In [ ]:
ISO = re.compile(r"^\d{4}-\d{2}-\d{2}T")
YMD = re.compile(r"^\d{4}-\d{2}-\d{2}$")
US  = re.compile(r"^\d{1,2}/\d{1,2}/\d{4}$")

def date_profile(s):
    sample = s.dropna().astype(str).head(2000)
    if len(sample) < 5: return None
    hits = {"ISO-8601": sample.str.match(ISO).sum(),
            "Y-m-d":    sample.str.match(YMD).sum(),
            "US m/d/Y": sample.str.match(US).sum()}
    total_hits = sum(hits.values())
    if total_hits / len(sample) < 0.8:   # <80% date-shaped -> not a date column
        return None
    return {k: v for k, v in hits.items() if v > 0}

rows = []
for c in df.columns:
    p = date_profile(df[c])
    if p:
        rows.append({"column": c, "n_formats": len(p),
                     "formats": " | ".join(f"{k}x{v}" for k, v in p.items()),
                     "fill_rate": f"{df[c].notna().mean():.1%}"})
dates = pd.DataFrame(rows).sort_values("n_formats", ascending=False)
print(f"True date columns detected: {len(dates)}  "
      f"(mixed-format: {(dates.n_formats > 1).sum()})")
dates

## 8. Numeric review

In [ ]:
num = df.select_dtypes(include=[np.number])
stats = num.describe().T[["count","mean","std","min","50%","max"]]
stats["null_%"] = ((1 - num.notna().mean())*100).round(1)
# flag suspicious columns
stats["flag"] = np.where(stats["max"] == stats["min"], "CONSTANT (dead column)",
                 np.where(stats["min"] < 0, "has negatives", ""))
suspicious = stats[(stats.flag != "") | (stats["null_%"] > 95)]
print(f"Numeric columns: {num.shape[1]} | flagged as suspicious: {len(suspicious)}")
suspicious.sort_values("null_%", ascending=False).head(25)

## 9. CDE candidate ranking — the shortlist for governance

In [ ]:
BUSINESS_HINTS = ("name","id","country","status","type","currency","date","number","code",
                  "sap","duns","owner","region","segment","sanction","denied","credit",
                  "revenue","email","phone","website","contract","tier","channel","parent")

rank = []
for c in df.columns:
    k = klass[c]
    if k in ("empty", "junk/UI", "constant"):        # never a CDE
        continue
    f = fill[c]
    name_hit = any(h in c.lower() for h in BUSINESS_HINTS)
    score = (f * 50) + (25 if name_hit else 0) + (15 if k in ("categorical","unique key") else 0) \
            + (10 if not c.endswith("__c") else 5)  # standard SF fields slightly favoured
    rank.append({"column": c, "class": k, "fill_rate": round(f, 3),
                 "distinct": int(card.get(c, 0)), "score": round(score, 1)})

cde_rank = pd.DataFrame(rank).sort_values("score", ascending=False).reset_index(drop=True)
print(f"Candidate pool: {len(cde_rank)} columns (after removing empty/junk/constant)")
print("\nTop 40 CDE candidates:")
cde_rank.head(40)

In [ ]:
top = cde_rank.head(25).iloc[::-1]
plt.figure(figsize=(9, 8))
sns.barplot(x="score", y="column", data=top, palette="viridis")
plt.title("Top 25 CDE candidates (heuristic score)"); plt.xlabel("score"); plt.ylabel("")
plt.tight_layout(); plt.show()

## 10. Completeness of the CDE shortlist

In [ ]:
CDE = [c for c in ["Id","Name","Type","Customer_Status__c","CurrencyIsoCode","OwnerId",
       "BillingCountry","Country_Region__c","Country_Sub_Region__c","Region_Name__c",
       "SAP_Account_Name__c","REPORT_ACCOUNT_NAME__c","SAP_Sold_To__c","Duns_Number__c",
       "Website","Phone","US_Govt_Sanctioned__c","Denied_Party_Status__c",
       "Customer_Segmentation__c","Corporate_Account_Type__c","Service_Level__c",
       "Sales_Channel__c","CreatedDate","Contract_End_Date__c"] if c in df.columns]

cf = (df[CDE].notna().mean()*100).round(1).sort_values()
plt.figure(figsize=(9, 8))
colors = ["#cf3b3b" if v < 50 else "#c98510" if v < 90 else "#1f9d57" for v in cf.values]
sns.barplot(x=cf.values, y=cf.index, palette=colors)
plt.title("Completeness of CDE shortlist (% populated, full dataset)")
plt.xlabel("% populated"); plt.xlim(0, 105); plt.ylabel("")
for i, v in enumerate(cf.values): plt.text(v+1, i, f"{v:.0f}%", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 11. Violation-volume estimate (sanity-check the 30–40% assumption)

In [ ]:
viol = pd.Series(False, index=df.index)   # rows with >=1 violation
detail = []
def rule(name, mask):
    global viol
    m = mask.reindex(df.index, fill_value=False)
    detail.append({"rule": name, "violations": int(m.sum())})
    viol = viol | m

if "Name" in df: rule("Name is null", df["Name"].isna())
if len(nv) >= 2:
    m = df[nv].notna().all(axis=1) & (df[nv].nunique(axis=1) > 1)
    rule("Name fields inconsistent", m)
if "Duns_Number__c" in df:
    d = df["Duns_Number__c"].astype(str).str.replace(r"\.0$", "", regex=True)
    rule("DUNS not 9 digits", df["Duns_Number__c"].notna() & ~d.str.fullmatch(r"\d{9}"))
if "Website" in df:
    rule("Website malformed",
         df["Website"].notna() & ~df["Website"].astype(str).str.match(r"^(https?://|www\.)", case=False))
for c in ["Customer_Segmentation__c","Corporate_Account_Type__c","Sales_Channel__c"]:
    if c in df: rule(f"{c} missing", df[c].isna())
if "Pipeline_Amount_Total__c" in df:
    rule("Pipeline amount negative", pd.to_numeric(df["Pipeline_Amount_Total__c"], errors="coerce") < 0)

est = pd.DataFrame(detail).sort_values("violations", ascending=False)
est["% of rows"] = (est.violations / n_rows * 100).round(1)
display(est)

print(f"\nTotal violation records (sum)   : {est.violations.sum():,}")
print(f"Distinct rows with >=1 violation : {viol.sum():,}  ({viol.mean():.1%} of all rows)")
if "Customer_Status__c" in df:
    print(f"Same, excluding Temporary        : {(viol & ~temp_mask).sum():,} "
          f"({(viol & ~temp_mask).sum()/max((~temp_mask).sum(),1):.1%} of in-scope rows)")

plt.figure(figsize=(9, 4.5))
sns.barplot(x="violations", y="rule", data=est, palette="rocket")
plt.title("Estimated violations by rule (only a handful of rules)")
plt.xlabel("records"); plt.ylabel(""); plt.tight_layout(); plt.show()

> This uses only ~8 rules. A production rule set (20–40 CDEs × several dimensions) will push the
> per-row violation rate substantially higher — which is what makes Karthik's **30–40%** estimate credible,
> and why violations must be stored as a **worklist**, not just counted.

## 12. Column pruning benchmark — the scaling fix

In [ ]:
usecols = [c for c in CDE if c in df.columns]
df_pruned = pd.read_csv(CSV_PATH, usecols=usecols, low_memory=False)
mem_pruned = df_pruned.memory_usage(deep=True).sum()/1024**2

print(f"All {n_cols} columns : {mem_mb:>9,.1f} MB")
print(f"Pruned to {len(usecols):>3}   : {mem_pruned:>9,.1f} MB")
print(f"Saving              : {(1-mem_pruned/mem_mb):.1%}\n")

pr = mem_pruned*1024/len(df_pruned)
cmp_df = pd.DataFrame({"rows":[1_000_000, 5_000_000, 10_000_000]})
cmp_df["all_columns_GB"] = (cmp_df["rows"]*per_row_kb/1024**2).round(1)
cmp_df["pruned_GB"]      = (cmp_df["rows"]*pr/1024**2).round(1)
display(cmp_df)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(cmp_df)); w = .38
ax.bar(x-w/2, cmp_df.all_columns_GB, w, label=f"all {n_cols} columns", color="#cf3b3b")
ax.bar(x+w/2, cmp_df.pruned_GB, w, label=f"pruned to {len(usecols)}", color="#1f9d57")
ax.axhline(16, ls="--", color="#141a22", lw=1); ax.text(0, 17, "typical 16 GB laptop", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([f"{r:,}" for r in cmp_df.rows])
ax.set_ylabel("estimated RAM (GB)"); ax.set_xlabel("rows")
ax.set_title("Memory: all columns vs pruned"); ax.legend()
plt.tight_layout(); plt.show()

## 13. Missingness pattern (row-sampled for speed)

In [ ]:
sample = df_pop.sample(min(500, len(df_pop)), random_state=0)
show = [c for c in df_pop.columns if c not in junk][:60]
plt.figure(figsize=(13, 5))
sns.heatmap(sample[show].isna(), cbar=False, cmap=["#1f9d57", "#f2f2f2"])
plt.title("Missingness pattern — 500 sampled rows x first 60 real columns "
          "(green = present, grey = missing)")
plt.xlabel("columns"); plt.ylabel("sampled rows"); plt.xticks(rotation=90, fontsize=6)
plt.tight_layout(); plt.show()

## 14. Summary

In [ ]:
summary = pd.DataFrame([
 ("Rows x Columns",            f"{n_rows:,} x {n_cols}"),
 ("Memory (all columns)",      f"{mem_mb:,.0f} MB  ({per_row_kb:.1f} KB/row)"),
 ("Projected @5M rows",        f"{5_000_000*per_row_kb/1024**2:,.0f} GB  -> pruning required"),
 ("Empty columns",             f"{len(fully_empty)}"),
 ("Junk/UI columns",           f"{len(junk_s)}"),
 ("Constant columns",          f"{sum(1 for v in klass.values() if v=='constant')}"),
 ("Real CDE candidate pool",   f"{len(cde_rank)}"),
 ("Duplicate Id",              f"{int(df['Id'].duplicated().sum()) if 'Id' in df else 'n/a'}  (key is safe)"),
 ("Rows w/ >=1 violation",     f"{viol.sum():,} ({viol.mean():.1%}) from only ~8 rules"),
], columns=["metric", "value"])
summary

### Findings → decisions

| # | Finding | Decision needed |
|---|---|---|
| 1 | ~13 KB/row across 443 cols → **5M rows won't fit in RAM** | Prune to CDE columns + use DuckDB/batching |
| 2 | Junk HTML/UI columns eat a large share of memory | Exclude from the element catalog entirely |
| 3 | **~62% of rows are `Temporary`** | Exclude from scope, or expose as a dashboard filter |
| 4 | **Name fields disagree ~50%** of the time | Flagship consistency rule — which name is authoritative? |
| 5 | Near-duplicate categories (`Middle East` / `MiddleEast`) | `allowed_values` rules + a cleanup list |
| 6 | `Id` has **0 duplicates** at 157K | Confirmed as the violation `record_key` |
| 7 | Dead numeric columns (all-zero) & negatives | Drop dead ones; range rules on the rest |
| 8 | Small sample badly misleads (`Type`: 2 vs 34 values) | Profile on a **large** sample before suggesting rules |

**Nothing here breaks the framework design** — it adds a *scope filter*, confirms the *record key*,
and makes *column pruning* mandatory at ingestion.